In [1]:
import pandas as pd
import glob
import os
import re

In [2]:
#Define function to loop through dashboard data files

def load_dashboard_file(path):
    filename = os.path.basename(path).lower()

    #extract year
    year_match = re.search(r"(\d{4})",filename)
    year = year_match.group(1) if year_match else None

    #extract indicator
    indicator = filename.replace(".txt","")
    if year:
        indicator = indicator.replace(f"_{year}","")
    df = pd.read_csv(path, sep="\t", dtype=str, encoding="latin1")
    df.columns = df.columns.str.lower().str.strip()

    df["indicator"] = indicator
    df["year"] = year
    return df

In [3]:
all_dfs = [load_dashboard_file(f) for f in glob.glob("../data/raw/*.txt")]
full_df = pd.concat(all_dfs, ignore_index=True)

In [4]:
#inspect the dataframe
full_df.shape
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3545437 entries, 0 to 3545436
Data columns (total 62 columns):
 #   Column                     Dtype 
---  ------                     ----- 
 0   cds                        object
 1   rtype                      object
 2   schoolname                 object
 3   districtname               object
 4   countyname                 object
 5   charter_flag               object
 6   coe_flag                   object
 7   dass_flag                  object
 8   studentgroup               object
 9   currnumer                  object
 10  currdenom                  object
 11  currstatus                 object
 12  priornumer                 object
 13  priordenom                 object
 14  priorstatus                object
 15  change                     object
 16  safetynet                  object
 17  statuslevel                object
 18  changelevel                object
 19  color                      object
 20  box                     

In [5]:
#drop all rows expect those with "Aspire" in the school name
aspire_df = full_df[full_df["schoolname"].str.contains("Aspire", case=False, na=False)]

In [6]:
aspire_df.head()

,cds,rtype,schoolname,districtname,countyname,charter_flag,coe_flag,dass_flag,studentgroup,currnumer,...,currprate_tested,currprate,currnumprloss,priorprate_enrolled,priorprate_tested,priorprate,priornumprloss,priordenom_withoutprloss,priorstatus_withoutprloss,type
1568,34674390102343,S,Aspire Capitol Heights Academy,Sacramento City Unified,Sacramento,Y,NaN,NaN,ALL,55,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1569,34674390102343,S,Aspire Capitol Heights Academy,Sacramento City Unified,Sacramento,Y,NaN,NaN,AI,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1570,34674390102343,S,Aspire Capitol Heights Academy,Sacramento City Unified,Sacramento,Y,NaN,NaN,AS,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1571,34674390102343,S,Aspire Capitol Heights Academy,Sacramento City Unified,Sacramento,Y,NaN,NaN,AA,39,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1572,34674390102343,S,Aspire Capitol Heights Academy,Sacramento City Unified,Sacramento,Y,NaN,NaN,SWD,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
#Check for any schools with "Aspire" that are not Aspire Charters
aspire_df["schoolname"].unique()[:100]

array(['Aspire Capitol Heights Academy', 'Aspire Rosa Parks Academy',
       'Aspire Antonio Maria Lugo Academy',
       'Aspire Berkley Maynard Academy',
       'Aspire Ollin University Preparatory Academy',
       'Aspire Summit Charter Academy', 'Aspire Port City Academy',
       'Aspire Junior Collegiate Academy', 'Aspire University Charter',
       'Aspire Golden State College Preparatory Academy',
       'Aspire Langston Hughes Academy', 'Aspire ERES Academy',
       'Aspire Vanguard College Preparatory Academy',
       'Aspire Alexander Twilight College Preparatory Academy',
       'Aspire Titan Academy',
       'Aspire Alexander Twilight Secondary Academy',
       'Aspire APEX Academy', 'Aspire Gateway Academy Charter',
       'Aspire Firestone Academy Charter', 'Aspire Pacific Academy',
       'Aspire Slauson Academy Charter',
       'Aspire Juanita Tate Academy Charter',
       'Aspire Inskeep Academy Charter',
       'Aspire Centennial College Preparatory Academy',
       'A

In [8]:
#remove ERES and Aspire Community Day Rows
aspire_df = aspire_df[
    ~aspire_df["schoolname"].str.contains("ERES|Community Day|North State", case=False, na=False)]

In [10]:
aspire_df["schoolname"].nunique()

38

In [11]:
aspire_df["schoolname"].value_counts()[aspire_df["schoolname"].value_counts() > 1]

schoolname
Aspire Langston Hughes Academy                           379
Aspire Alexander Twilight Secondary Academy              376
Aspire Rosa Parks Academy                                361
Aspire East Palo Alto Charter                            358
Aspire APEX Academy                                      355
Aspire Vanguard College Preparatory Academy              355
Aspire Richmond Technology Academy                       354
Aspire River Oaks Charter                                353
Aspire Richmond Ca. College Preparatory Academy          350
Aspire Port City Academy                                 348
Aspire Alexander Twilight College Preparatory Academy    348
Aspire Berkley Maynard Academy                           344
Aspire Summit Charter Academy                            343
Aspire Golden State College Preparatory Academy          338
Aspire Capitol Heights Academy                           337
Aspire Vincent Shalvey Academy                           330
Aspire Univer

In [12]:
#merge Aspire Stockton Tk-5 data with Aspire Arts & Sciences (they are the same school - changed names)
aspire_df["schoolname"] = aspire_df["schoolname"].replace({
    "Aspire Stockton TK-5 Elementary Academy": "Aspire Arts & Sciences Academy"
})

In [13]:
#merge Ben Holt Middle data (same school) 
aspire_df["schoolname"] = aspire_df["schoolname"].replace({
    "Aspire Benjamin Holt Middle School": "Aspire Benjamin Holt Middle"
})

In [15]:
#check to see if 36 school names exist
aspire_df["schoolname"].nunique()

36

In [17]:
#drop columns we do not need
cols_to_keep = [
    "cds", "schoolname", "indicator", "year", "studentgroup",
    "currstatus", "priorstatus", "change",
    "statuslevel", "changelevel", "color"
]

aspire_clean = aspire_df[cols_to_keep].copy()

In [19]:
aspire_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11217 entries, 1568 to 3534448
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   cds           11217 non-null  object
 1   schoolname    11217 non-null  object
 2   indicator     11217 non-null  object
 3   year          11217 non-null  object
 4   studentgroup  11217 non-null  object
 5   currstatus    7444 non-null   object
 6   priorstatus   6103 non-null   object
 7   change        5975 non-null   object
 8   statuslevel   11217 non-null  object
 9   changelevel   9398 non-null   object
 10  color         9398 non-null   object
dtypes: object(11)
memory usage: 1.0+ MB


In [24]:
aspire_clean.head()

,cds,schoolname,indicator,year,studentgroup,currstatus,priorstatus,change,statuslevel,changelevel,color
1568,34674390102343,Aspire Capitol Heights Academy,chronicdownload2018,2018,ALL,19,16.6,2.5,2,2,2
1569,34674390102343,Aspire Capitol Heights Academy,chronicdownload2018,2018,AI,NaN,NaN,NaN,0,0,0
1570,34674390102343,Aspire Capitol Heights Academy,chronicdownload2018,2018,AS,NaN,NaN,NaN,0,0,0
1571,34674390102343,Aspire Capitol Heights Academy,chronicdownload2018,2018,AA,22.2,17.9,4.3,1,1,1
1572,34674390102343,Aspire Capitol Heights Academy,chronicdownload2018,2018,SWD,20,13.2,6.8,2,2,2


In [29]:
#clean up indicator column
indicator_map = {
    # Chronic Absenteeism
    "chronicdownload2018": "chronic_absenteeism",
    "chronicdownload2019": "chronic_absenteeism",
    "chronicdownload2022": "chronic_absenteeism",
    "chronicdownload2023": "chronic_absenteeism",
    "chronicdownload2024": "chronic_absenteeism",
    "chronicdownload2025": "chronic_absenteeism",

    # ELA SBAC
    "eladownload2018": "ela_sbac",
    "eladownload2019": "ela_sbac",
    "eladownload2022": "ela_sbac",
    "eladownload2023": "ela_sbac",
    "eladownload2024": "ela_sbac",
    "eladownload2025": "ela_sbac",

    # Math SBAC
    "mathdownload2018": "math_sbac",
    "mathdownload2019": "math_sbac",
    "mathdownload2022": "math_sbac",
    "mathdownload2023": "math_sbac",
    "mathdownload2024": "math_sbac",
    "mathdownload2025": "math_sbac",

    # Suspension Rate
    "suspdownload2018": "suspension_rate",
    "suspdownload2019": "suspension_rate",
    "suspdownload2022": "suspension_rate",
    "suspdownload2023": "suspension_rate",
    "suspdownload2024": "suspension_rate",
    "suspdownload2025": "suspension_rate",}

In [30]:
aspire_clean["indicator"] = aspire_clean["indicator"].replace(indicator_map)

In [31]:
aspire_clean["indicator"].value_counts()

indicator
ela_sbac               3088
math_sbac              3087
chronic_absenteeism    2553
suspension_rate        2489
Name: count, dtype: int64

In [32]:
aspire_clean.head()

,cds,schoolname,indicator,year,studentgroup,currstatus,priorstatus,change,statuslevel,changelevel,color
1568,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,ALL,19,16.6,2.5,2,2,2
1569,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,AI,NaN,NaN,NaN,0,0,0
1570,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,AS,NaN,NaN,NaN,0,0,0
1571,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,AA,22.2,17.9,4.3,1,1,1
1572,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,SWD,20,13.2,6.8,2,2,2


In [36]:
#map student group abbreviations to full student group names
studentgroup_map = {
    "ALL": "All Students",
    "AA": "Black or African American",
    "AI": "American Indian or Alaska Native",
    "AS": "Asian",
    "FI": "Filipino",
    "HI": "Hispanic or Latino",
    "PI": "Native Hawaiian or Pacific Islander",
    "WH": "White",
    "MR": "Multiple Races/Two or More Races",

    # English Learner categories
    "EL": "English Learner",
    "LTEL": "Long-Term English Learner",
    "ELO": "English Learners Only",
    "RFP": "Recently Reclassified Fluent-English Proficient Only",
    "EO": "English Only",

    # Programmatic groups
    "SED": "Socioeconomically Disadvantaged",
    "SWD": "Students with Disabilities",
    "FOS": "Foster Youth",
    "HOM": "Homeless Youth",

    # Assessment participation groups
    "SBA": "Students who took the Smarter Balanced Summative Assessment",
    "CAA": "Students who took the California Alternate Assessment",
}

In [37]:
aspire_clean["studentgroup"] = aspire_clean["studentgroup"].replace(studentgroup_map)

In [38]:
aspire_clean.head()

,cds,schoolname,indicator,year,studentgroup,currstatus,priorstatus,change,statuslevel,changelevel,color
1568,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,All Students,19,16.6,2.5,2,2,2
1569,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,American Indian or Alaska Native,NaN,NaN,NaN,0,0,0
1570,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,Asian,NaN,NaN,NaN,0,0,0
1571,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,Black or African American,22.2,17.9,4.3,1,1,1
1572,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,Students with Disabilities,20,13.2,6.8,2,2,2


In [39]:
aspire_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11217 entries, 1568 to 3534448
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   cds           11217 non-null  object
 1   schoolname    11217 non-null  object
 2   indicator     11217 non-null  object
 3   year          11217 non-null  object
 4   studentgroup  11217 non-null  object
 5   currstatus    7444 non-null   object
 6   priorstatus   6103 non-null   object
 7   change        5975 non-null   object
 8   statuslevel   11217 non-null  object
 9   changelevel   9398 non-null   object
 10  color         9398 non-null   object
dtypes: object(11)
memory usage: 1.0+ MB


In [42]:
aspire_clean["studentgroup"].unique()

array(['All Students', 'American Indian or Alaska Native', 'Asian',
       'Black or African American', 'Students with Disabilities',
       'English Learner', 'Filipino', 'Foster Youth',
       'Hispanic or Latino', 'Homeless Youth',
       'Native Hawaiian or Pacific Islander',
       'Socioeconomically Disadvantaged',
       'Multiple Races/Two or More Races', 'White',
       'Long-Term English Learner', 'English Learners Only',
       'English Only',
       'Recently Reclassified Fluent-English Proficient Only',
       'Students who took the California Alternate Assessment',
       'Students who took the Smarter Balanced Summative Assessment'],
      dtype=object)

In [43]:
#move "All Students" group to separate dataframe
aspire_all = aspire_clean[aspire_clean["studentgroup"] == "All Students"].copy()

In [44]:
aspire_all.head()

,cds,schoolname,indicator,year,studentgroup,currstatus,priorstatus,change,statuslevel,changelevel,color
1568,34674390102343,Aspire Capitol Heights Academy,chronic_absenteeism,2018,All Students,19,16.6,2.5,2,2,2
4494,39686760108647,Aspire Rosa Parks Academy,chronic_absenteeism,2018,All Students,13.9,8,6,2,1,1
5306,19101990109660,Aspire Antonio Maria Lugo Academy,chronic_absenteeism,2018,All Students,4.4,5.1,-0.7,4,4,4
5404,01612590109819,Aspire Berkley Maynard Academy,chronic_absenteeism,2018,All Students,16.8,11.4,5.4,2,1,1
7112,19101990112128,Aspire Ollin University Preparatory Academy,chronic_absenteeism,2018,All Students,6.1,7.6,-1.5,3,4,4


In [45]:
aspire_all.info()

<class 'pandas.core.frame.DataFrame'>
Index: 842 entries, 1568 to 3534436
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   cds           842 non-null    object
 1   schoolname    842 non-null    object
 2   indicator     842 non-null    object
 3   year          842 non-null    object
 4   studentgroup  842 non-null    object
 5   currstatus    842 non-null    object
 6   priorstatus   693 non-null    object
 7   change        693 non-null    object
 8   statuslevel   842 non-null    object
 9   changelevel   699 non-null    object
 10  color         699 non-null    object
dtypes: object(11)
memory usage: 78.9+ KB


In [46]:
# Convert continuous numeric fields
aspire_all["currstatus"] = pd.to_numeric(aspire_all["currstatus"], errors="coerce")
aspire_all["priorstatus"] = pd.to_numeric(aspire_all["priorstatus"], errors="coerce")
aspire_all["change"] = pd.to_numeric(aspire_all["change"], errors="coerce")

# Convert categorical numeric fields
aspire_all["statuslevel"] = pd.to_numeric(aspire_all["statuslevel"], errors="coerce").astype("Int64")
aspire_all["changelevel"] = pd.to_numeric(aspire_all["changelevel"], errors="coerce").astype("Int64")
aspire_all["color"] = pd.to_numeric(aspire_all["color"], errors="coerce").astype("Int64")

# Convert year to integer
aspire_all["year"] = aspire_all["year"].astype(int)

In [47]:
aspire_all.info()

<class 'pandas.core.frame.DataFrame'>
Index: 842 entries, 1568 to 3534436
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   cds           842 non-null    object 
 1   schoolname    842 non-null    object 
 2   indicator     842 non-null    object 
 3   year          842 non-null    int64  
 4   studentgroup  842 non-null    object 
 5   currstatus    842 non-null    float64
 6   priorstatus   693 non-null    float64
 7   change        693 non-null    float64
 8   statuslevel   842 non-null    Int64  
 9   changelevel   699 non-null    Int64  
 10  color         699 non-null    Int64  
dtypes: Int64(3), float64(3), int64(1), object(4)
memory usage: 81.4+ KB
